## Load Environment Variables

In [1]:
import os, httpx
from dotenv import load_dotenv

load_dotenv("/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/.env", override=True)

CA_BUNDLE = "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/ca-bundle.pem"
os.environ["SSL_CERT_FILE"] = CA_BUNDLE
os.environ["REQUESTS_CA_BUNDLE"] = CA_BUNDLE
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "eli5-experiment"

http_client = httpx.Client(verify=CA_BUNDLE)
http_async_client = httpx.AsyncClient(verify=CA_BUNDLE)

## Create AI Application

### Setup 

As usual, let's define our prompt and give our application access to the web.

In [2]:
# Initialize web search tool
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=1)

# Define prompt template
prompt = """You are a professor and expert in explaining complex topics in a way that is easy to understand. 
Your job is to answer the provided question so that even a 5 year old can understand it. 
You have provided with relevant background context to answer the question.

Question: {question} 

Context: {context}

Answer:"""

/var/folders/wr/xj2yx28s7xvg8trnrb4nc5_w0000gn/T/ipykernel_56345/2405568056.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=1)


### Define Application Logic

The logic here is the same as in the tracing module. We define a search step to scan the web, and an explain step for an LLM to summarize the web results.

In [3]:
from groq import Groq
from langsmith import traceable

# Create Application
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in web_docs])
    return web_results
    
@traceable(run_type="llm")
def explain(question, context):
    formatted = prompt.format(question=question, context=context)
    
    completion = groq_client.chat.completions.create(
        messages=[
            {"role": "system", "content": formatted},
            {"role": "user", "content": question},
        ],
        model="llama-3.3-70b-versatile",
    )
    return completion.choices[0].message.content

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer

## Setup for Experiment

Now we're ready to run experiments, and test our application's performance on our dataset.

### Import LangSmith Client

First, we'll create a LangSmith client to use the SDK, and specify the dataset we'd like to run our experiment on.

### Upload Dataset to LangSmith
Before running our experiment, we need to upload our dataset to LangSmith. We'll read the CSV file and create a dataset with the name "eli5-golden".

In [4]:
from langsmith import Client

client = Client()
dataset_name = "eli5-golden"

In [6]:
import pandas as pd
from langsmith import Client

# Read the dataset CSV file
df = pd.read_csv("dataset.csv")

# Convert to LangSmith format
examples = []
for _, row in df.iterrows():
    examples.append({
        "inputs": {"question": row["input_question"]},
        "outputs": {"output": row["output_output"]}
    })

# Create dataset in LangSmith
try:
    # Try to read the dataset first to see if it already exists
    existing_dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' already exists with {len(list(client.list_examples(dataset_name=dataset_name)))} examples")
except:
    # Dataset doesn't exist, create it
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="ELI5 (Explain Like I'm 5) dataset for evaluating AI explanations"
    )
    
    # Add examples to the dataset
    client.create_examples(
        inputs=[ex["inputs"] for ex in examples],
        outputs=[ex["outputs"] for ex in examples],
        dataset_id=dataset.id
    )
    
    print(f"Successfully created dataset '{dataset_name}' with {len(examples)} examples")


Successfully created dataset 'eli5-golden' with 9 examples


### Define Evaluators

#### Custom Code Evaluator

We'll first define a custom code evaluator, which are useful to measure deterministic or close-ended metrics. 

In [7]:
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200

This particular custom code evaluator is a simple Python function that checks if our application produces outputs that are less than or equal to 200 words long.

#### LLM-as-a-Judge Evaluator

For open-ended metrics, it's can be powerful to use an LLM to score the outputs.

Let's use an LLM to check whether our application produces correct outputs. First, let's define a scoring schema for our LLM to adhere to in its response.

In [8]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score of the correctness of the answer, from 0 to 1")

We'll define a function to give an LLM our application's outputs, alongside the reference outputs stored in our dataset. 

The LLM will then be able to reference the "right" output to judge if our application's answer meets our accuracy standards.

In [9]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

    <Rubric>
        A correct answer:
        - Provides accurate information
        - Uses suitable analogies and examples
        - Contains no factual errors
        - Is logically consistent

        When scoring, you should penalize:
        - Factual errors
        - Incoherent analogies and examples
        - Logical inconsistencies
    </Rubric>

    <Instructions>
        - Carefully read the input and output
        - Use the reference output to determine if the model output contains errors
        - Focus whether the model output uses accurate analogies and is logically consistent
    </Instructions>

    <Reminder>
        The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Use the reference outputs below to help you evaluate the correctness of the response:
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["question"], outputs["output"], reference_outputs["output"])
    structured_llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
        http_client=http_client,
        http_async_client=http_async_client,
    ).with_structured_output(CorrectnessScore)
    generation = structured_llm.invoke([HumanMessage(content=prompt)])
    return generation.score == 1

### Define Run Function

We'll define a function to run our application on the example inputs of our dataset. This is function that will be called when we run our experiment.

In [10]:
# 4. Define a function to run your application
def run(inputs: dict):
    return eli5(inputs["question"])

## Run Experiment

We have all the necessary components, so let's run our experiment! 

In [11]:
from langsmith import evaluate

evaluate(
    run,
    data=dataset_name,
    evaluators=[correctness, conciseness],
    experiment_prefix="eli5-llama-3.3-70b"
)

View the evaluation results for experiment: 'eli5-llama-3.3-70b-13b6a88e' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/1b5a90ec-2f8e-451d-9715-56ce45dbc24f/compare?selectedSessions=b974693d-17c4-4d6a-8ae2-b2709ca66380




0it [00:00, ?it/s]

Error running evaluator <DynamicRunEvaluator correctness> on run 019cc48c-7db2-7160-8ffe-7e3a85418baa: BadRequestError('Error code: 400 - {\'error\': {\'message\': \'tool call validation failed: parameters for tool CorrectnessScore did not match schema: errors: [`/score`: expected integer, but got number]\', \'type\': \'invalid_request_error\', \'code\': \'tool_use_failed\', \'failed_generation\': \'<function=CorrectnessScore>{"score": 0.9} </function>\'}}')
Traceback (most recent call last):
  File "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/lc-academy-env/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1601, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph--

,inputs.question,outputs.output,error,reference.output,feedback.wrapper,feedback.conciseness,execution_time,example_id,id
0,How does string theory work?,"Oh boy, are you ready for a cool idea? \n\nIma...",None,"Okay! Imagine that everything in the universe,...",None,False,3.273047,09bda6e6-9470-4b96-9acb-84700a01b4a4,019cc48c-7db2-7160-8ffe-7e3a85418baa
1,How does a democracy work?,"Oh boy, let me explain it to you in a super si...",None,Okay! Imagine you and your friends want to dec...,None,False,3.176718,1015931b-54bd-4321-af8c-ffb423a0b492,019cc48c-8b86-7a82-bc4b-f167f6769a97
2,What is LangSmith by LangChain?,"Imagine you have a lot of toys, and each toy c...",None,Okay! Imagine you have a big box of toys that ...,None,True,2.399338,13468ea6-5ec9-4828-82b7-2a33ea5570eb,019cc48c-98b4-73b1-ba7f-20ddf20ea333
3,What is trustcall library?,"Imagine you have a lot of toys, and each toy h...",None,"Alright, imagine you have a toy box where each...",None,True,2.707107,6a3e0ad3-1c29-4d29-a432-6268e24b4104,019cc48c-a2d3-7dd1-a545-10ccd7be5718
4,What is the Langchain framework?,"Imagine you have a lot of different toys, like...",None,Okay! Imagine you want to build a really cool ...,None,True,2.811724,715f20e7-b181-4b00-8a1b-4bc2713e66f2,019cc48c-ae1d-75c2-a12f-46222bc268c7
5,How does photosynthesis work?,"Oh boy, are you ready for a cool explanation? ...",None,"Okay! Imagine plants are like tiny chefs, and ...",None,True,2.954349,795dabce-4a53-489d-a300-75395cd7f591,019cc48c-ba1d-7873-860e-b937a1c97b07
6,What is sound?,"So, you know how sometimes you can hear things...",None,Okay! Imagine you have a drum. When you hit it...,None,True,3.018017,a5d0d7b7-976a-4914-81f9-cd858ab06469,019cc48c-c65a-7cf1-962b-81c9c626f0be
7,What is LangGraph?,"Imagine you have a lot of toys, and each toy c...",None,"Okay, imagine you have a big box of LEGO brick...",None,True,2.726055,e5209283-153a-4c5b-8c4b-9d52959beb47,019cc48c-d2ef-7fb1-a5a5-437e07786f5c
8,Why is the sky blue?,"Oh boy, let me tell you a secret about the sky...",None,Alright! Imagine the sky is like a big bowl of...,None,False,2.850801,f4833a05-4995-41b6-85ca-96823ad487b8,019cc48c-de4e-7ba0-bd32-d2209f6315d4
